# Food Delivery AI Agent — Interim Report

**Course project · LLM-powered SQL analytics over a food-delivery database**

This notebook is organised to match the interim-report rubric one-to-one:

| # | Section | Marks |
|---|---------|-------|
| 1 | Loading and Setting Up the LLM | 8 |
| 2 | Question Answering LLM | 10 |
| 3 | Build SQL Agent | 16 |
| 4 | Business Report Quality | 6 |

**LLM backend:** OpenAI (GPT) via LangChain.  
**Database:** `data/food_delivery.db` — a synthetic SQLite database of 300 orders across customers, restaurants, delivery agents and order line-items, generated reproducibly by `create_database.py`.

> **Before running:** create a `.env` file from `.env.example` and paste your `OPENAI_API_KEY`. Then run the cells top to bottom.

---
## 1. Loading and Setting Up the LLM  *(8 Marks)*

**Goals:** import & initialise libraries · configure the base LLM · load environment variables / API keys · fix configs so results are reproducible.

### 1.1 Import and initialise required libraries

In [ ]:
import os
import sqlite3

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent

print('Libraries imported successfully.')

### 1.2 Set up environment variables and API keys

We never hard-code secrets. The key lives in a local `.env` file (git-ignored) and is loaded at runtime by `python-dotenv`.

In [ ]:
load_dotenv()  # reads .env in the project root

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
MODEL_NAME = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')

assert OPENAI_API_KEY, (
    'OPENAI_API_KEY not found. Copy .env.example to .env and add your key.'
)
print(f'API key loaded. Using model: {MODEL_NAME}')

### 1.3 Configure the base LLM (with fixed parameters for reproducibility)

`temperature=0` makes generation as deterministic as possible, and we pass a fixed `seed` so repeated runs return stable answers — this is what the rubric means by *fixing configs and parameters*.

In [ ]:
llm = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0,        # deterministic / reproducible outputs
    seed=42,              # fixed seed for repeatability
    max_retries=2,
)

# Smoke test: confirm the model responds.
print(llm.invoke('Reply with exactly: LLM is ready.').content)

---
## 2. Question Answering LLM  *(10 Marks)*

**Goals:** give the LLM sample questions to validate query understanding · comment on accuracy & clarity · refine the prompts/formatting · re-run the same questions · comment on the improved responses.

### 2.1 Baseline — sample questions with plain prompts

We start with three questions a food-delivery business analyst might ask, phrased naturally and sent with **no special instructions**.

In [ ]:
sample_questions = [
    'What is a food delivery SQL agent and what can it do?',
    'Give me 3 KPIs a food delivery business should track.',
    'How would you find the highest-spending customer from an orders table?',
]

baseline_answers = []
for q in sample_questions:
    resp = llm.invoke(q).content
    baseline_answers.append(resp)
    print('Q:', q)
    print('A:', resp)
    print('-' * 80)

### 2.2 Comment on response accuracy and clarity (baseline)

*Fill in after running — example observations:*

- **Accuracy:** Answers are factually correct but generic. The KPI question, for instance, returns broad metrics not tailored to *our* schema.
- **Clarity:** Responses are verbose and unstructured (long paragraphs), which makes them harder to scan.
- **Consistency:** Format varies question-to-question — sometimes prose, sometimes lists.

**Conclusion:** correct but not concise or schema-aware → refine the prompt.

### 2.3 Refine the prompts / input formatting

We add (a) a **system prompt** giving the model a role and our database context, and (b) explicit **output-format instructions** (concise, bulleted, tied to our tables). Same questions, better scaffolding.

In [ ]:
SYSTEM_PROMPT = (
    'You are a data analyst for a food-delivery company. '
    'The database has these tables: customers, restaurants, '
    'delivery_agents, orders, order_items. '
    'Answer concisely in at most 4 bullet points. '
    'Be specific to this schema and, where relevant, mention the exact '
    'table or column names.'
)

refined_answers = []
for q in sample_questions:
    messages = [SystemMessage(SYSTEM_PROMPT), HumanMessage(q)]
    resp = llm.invoke(messages).content
    refined_answers.append(resp)
    print('Q:', q)
    print('A:', resp)
    print('-' * 80)

### 2.4 Comment on response accuracy and clarity (refined)

*Fill in after running — example observations:*

- **Accuracy:** Answers now reference our actual tables/columns (`orders.total_amount`, `customers.customer_id`), so they are directly actionable against our schema.
- **Clarity:** Bullet-point limit forces concise, scannable output with a consistent structure across all three questions.
- **Relevance:** The role + schema context removes generic filler and keeps answers on-domain.

**Takeaway:** prompt engineering (role, schema context, format constraints) improved relevance and readability without changing the underlying model.

---
## 3. Build SQL Agent  *(16 Marks)*

**Goals:** load the database with `SQLDatabase` · define the SQL agent · test it by retrieving all columns for a given Order ID · verify the output against the real database.

### 3.1 Load the database using SQLDatabase

LangChain's `SQLDatabase` wraps the SQLite file via SQLAlchemy and exposes the schema to the agent.

In [ ]:
DB_PATH = os.path.join('data', 'food_delivery.db')
assert os.path.exists(DB_PATH), (
    f'{DB_PATH} not found. Run: python3 create_database.py'
)

db = SQLDatabase.from_uri(f'sqlite:///{DB_PATH}')

print('Dialect      :', db.dialect)
print('Usable tables:', db.get_usable_table_names())

Peek at the schema the agent will see:

In [ ]:
print(db.get_table_info(['orders']))

### 3.2 Define the SQL Agent

`create_sql_agent` builds a ReAct-style agent with a SQL toolkit: it can list tables, inspect schemas, write a query, run it, and self-correct on errors. `verbose=True` prints the agent's reasoning + the SQL it generates.

In [ ]:
sql_agent = create_sql_agent(
    llm=llm,
    db=db,
    agent_type='openai-tools',
    verbose=True,
)
print('SQL agent ready.')

### 3.3 Test — retrieve all columns for an Order ID

We ask the agent (in plain English) for **every** detail of Order **1001**. The agent must join `orders` with `customers`, `restaurants`, `delivery_agents`, and `order_items` on its own.

In [ ]:
question = (
    'Retrieve all available columns and details for order_id 1001, '
    'including the customer, restaurant, delivery agent, order status, '
    'payment method, total amount, and every item in the order.'
)

result = sql_agent.invoke({'input': question})
print('\n=== AGENT ANSWER ===')
print(result['output'])

### 3.4 Verify the SQL Agent's accuracy against the database

We run the equivalent query **directly** with `sqlite3` and compare. If the agent's natural-language answer matches this ground truth, it is accurate.

In [ ]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

print('--- orders + joined reference data for order 1001 ---')
row = cur.execute('''
    SELECT o.order_id, c.name, c.city, r.name, r.cuisine,
           a.name, a.vehicle_type, o.order_date, o.status,
           o.payment_method, o.total_amount, o.delivery_time_min
    FROM orders o
    JOIN customers c       ON c.customer_id   = o.customer_id
    JOIN restaurants r     ON r.restaurant_id = o.restaurant_id
    JOIN delivery_agents a ON a.agent_id      = o.agent_id
    WHERE o.order_id = 1001
''').fetchone()
cols = ['order_id','customer','city','restaurant','cuisine','agent',
        'vehicle','order_date','status','payment','total_amount','delivery_min']
for c, v in zip(cols, row):
    print(f'  {c:<14}: {v}')

print('\n--- order_items for order 1001 ---')
for it in cur.execute('''
    SELECT item_name, quantity, unit_price
    FROM order_items WHERE order_id = 1001''').fetchall():
    print('  ', it)
conn.close()

**Verification note** *(fill in after running):* The agent's answer should match the ground-truth row above field-for-field (same customer, restaurant, agent, status, payment, total amount and the same line-items). Matching confirms the agent generated correct SQL and read the database accurately.

---
## 4. Business Report Quality  *(6 Marks)*

*Adheres to the business-report checklist: title, objective, data overview, methodology, findings, recommendations, and conclusion.*

### Business Report — Food Delivery AI SQL Agent (Interim)

**1. Objective.** Build an LLM-powered agent that answers natural-language business questions over a food-delivery database, removing the need for analysts to hand-write SQL.

**2. Data overview.** A SQLite database (`food_delivery.db`) with five related tables — `customers` (60), `restaurants` (25), `delivery_agents` (20), `orders` (300) and `order_items` (794). Orders link customers, restaurants and agents and carry status, payment method, total amount and delivery time.

**3. Methodology.** (a) Configured a deterministic OpenAI LLM via LangChain (`temperature=0`, fixed seed) for reproducibility. (b) Validated raw question answering and improved it through prompt engineering (role + schema context + format constraints). (c) Built a SQL agent with `create_sql_agent` that inspects the schema, writes SQL, executes it and self-corrects.

**4. Findings.** The refined prompts produced more accurate, schema-aware and concise answers than the baseline. The SQL agent correctly resolved a multi-table request (all details for Order 1001) and its output matched a direct SQL query used as ground truth.

**5. Recommendations / next steps.** Add few-shot SQL examples for complex analytics, cache frequent queries, add guardrails to restrict the agent to read-only `SELECT` statements, and build a simple UI for business users.

**6. Conclusion.** The interim build demonstrates an accurate, reproducible natural-language-to-SQL workflow over the food-delivery data and is a sound foundation for the final deliverable.